# 8.3 · AdaBoost / Adaptive Boosting

> **课程定位 / Where this fits**
> 第 3 课，**Part 8 · 集成学习**。
> Lesson 3, **Part 8 · Ensemble Learning**.
>
> 前两课是 bagging（并行、降方差）。从这一课起进入 **boosting（串行、降偏差）**。**AdaBoost** 是第一个成功的 boosting 算法（1995），思想极其优雅：**串行训练一串"弱学习器"（如树桩），每训完一个，就加大它分错的样本的权重**，让下一个学习器**专攻难例**；最后按各学习器的准确率加权投票。它是理解 GBDT(8.4) 的前奏。
> The first two lessons were bagging (parallel, variance). From here we enter **boosting (sequential, bias)**. **AdaBoost** (1995) was the first successful boosting algorithm, with an elegant idea: **train weak learners (e.g. stumps) one after another, and after each, up-weight the samples it got wrong** so the next learner **focuses on the hard cases**; finally vote weighted by each learner's accuracy. It's the prelude to GBDT (8.4).
>
> 💼 **实战/面试视角**："AdaBoost 怎么工作 / 样本权重怎么更新 / 它和 GBDT 的关系" 是 boosting 入门必考。
> 💼 **Practical/interview angle:** "how AdaBoost works / sample-weight updates / its relation to GBDT" — boosting basics.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $w_i$ —— 第 $i$ 个样本的权重 / weight of sample $i$
> - $h_m$ —— 第 $m$ 个弱学习器 / the $m$-th weak learner
> - $\alpha_m$ —— 第 $m$ 个学习器的话语权（投票权重）/ its say (vote weight)
> - $\text{err}_m$ —— 第 $m$ 个学习器的加权错误率 / its weighted error

> 💡 **面试相关 / Interview-relevant**
> - "AdaBoost 的样本权重和学习器权重怎么更新"（出镜率 ★★★★★）
> - "AdaBoost 在最小化什么损失（指数损失）"（★★★★）
> - "为什么用弱学习器（树桩）"（★★★★）
> - "AdaBoost 对异常值敏感吗"（★★★★，敏感）
> - "AdaBoost vs GBDT 区别"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 boosting 串行纠错的思想（vs bagging 并行平均）。
   Understand boosting's sequential correction (vs bagging's parallel averaging).
2. **从零**实现 AdaBoost（样本重加权 + 学习器加权投票）。
   Implement AdaBoost from scratch (sample reweighting + weighted vote).
3. 看清样本权重如何"追着难例跑"。
   See how sample weights chase the hard examples.
4. 理解 AdaBoost = 最小化**指数损失**，及它对异常值敏感。
   Understand AdaBoost = minimizing exponential loss, and its outlier sensitivity.
5. 对照 sklearn 并和 GBDT 比较。
   Compare with sklearn and contrast with GBDT.

## 目录 / TOC
1. [先建直觉 + 算法 ⭐](#1)
2. [🚢 数据 + 从零实现 ⭐](#2)
3. [样本权重追难例（可视化）⭐](#3)
4. [指数损失 + 对异常值敏感 ⭐](#4)
5. [对照 sklearn + vs GBDT + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + 算法 ⭐ / Intuition & Algorithm

AdaBoost 像一个**会反省的学习小组**：每个组员（弱学习器）只比瞎猜强一点点（如一个树桩——只有一层的决策树）。但他们**轮流上场，每个人都重点补前一个人犯的错**——具体做法是给前一轮**分错的样本更高的权重**，逼下一个学习器更关注它们。最后大家**按各自的准确率加权投票**。
AdaBoost is like a **self-reflective study group**: each member (weak learner) is only slightly better than chance (e.g. a stump — a one-level tree). But they **take turns, each focusing on the previous member's mistakes** — concretely, by **up-weighting the samples the previous round got wrong**, forcing the next learner to attend to them. Finally everyone **votes weighted by their own accuracy**.

**算法（离散 AdaBoost，标签用 ±1）**：
**Algorithm (discrete AdaBoost, labels in ±1):**
1. 初始化样本权重 $w_i = 1/n$（一开始一视同仁）。
   Initialize weights $w_i = 1/n$.
2. 对 $m = 1\dots M$：
   For $m = 1\dots M$:
   - 在**加权数据**上训一个弱学习器 $h_m$，算它的**加权错误率** $\text{err}_m=\sum_{i} w_i\,[h_m(\mathbf{x}_i)\ne y_i]$。
     Train $h_m$ on **weighted data**; compute weighted error $\text{err}_m$.
   - 算它的**话语权** $\alpha_m = \frac12\ln\frac{1-\text{err}_m}{\text{err}_m}$（错误率越低，话语权越大）。
     Its say $\alpha_m = \frac12\ln\frac{1-\text{err}_m}{\text{err}_m}$ (lower error → larger say).
   - **更新权重**：分错的样本 $w_i\leftarrow w_i\,e^{\alpha_m}$（放大），分对的缩小，再归一化。
     **Update weights:** misclassified $w_i\leftarrow w_i\,e^{\alpha_m}$ (amplify), correct ones shrink, then normalize.
3. 最终预测：$\hat y = \text{sign}\big(\sum_m \alpha_m h_m(\mathbf{x})\big)$（话语权加权投票）。
   Final: $\hat y = \text{sign}\big(\sum_m \alpha_m h_m(\mathbf{x})\big)$ (say-weighted vote).


<a id="2"></a>
## 2. 数据 + 从零实现 ⭐ / Data & From Scratch

用 **Titanic**。从零实现上面的算法。注意两个工程点：标签转成 **±1**（AdaBoost 的标准形式），弱学习器用 **决策树桩**（`max_depth=1`，"弱"的经典选择）。
Using **Titanic**. We implement the algorithm above. Two engineering points: convert labels to **±1** (AdaBoost's standard form), and use a **decision stump** (`max_depth=1`, the classic "weak" learner).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median()); d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)
X, y01 = d[feat].values, d["survived"].values
y = np.where(y01 == 1, 1, -1)                        # 标签转 ±1 (AdaBoost 标准形式) / labels to ±1
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

class AdaBoostScratch:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
    def fit(self, X, y):
        n = len(X)
        w = np.full(n, 1/n)                          # 初始权重: 人人平等
        self.learners, self.alphas = [], []
        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1)         # 弱学习器=树桩
            stump.fit(X, y, sample_weight=w)                    # 在加权数据上训
            pred = stump.predict(X)
            err = w[pred != y].sum()                            # 加权错误率
            err = min(max(err, 1e-10), 1 - 1e-10)              # 防 log(0)/除0
            alpha = 0.5 * np.log((1 - err) / err)              # 话语权: 错越少权越大
            w = w * np.exp(-alpha * y * pred)                  # 更新: 分错(y≠pred)→放大, 分对→缩小
            w = w / w.sum()                                     # 归一化
            self.learners.append(stump); self.alphas.append(alpha)
        return self
    def decision(self, X):
        # 各学习器预测按话语权 alpha 加权求和 / weighted sum of learners
        return sum(a * h.predict(X) for a, h in zip(self.alphas, self.learners))
    def predict(self, X):
        return np.where(self.decision(X) >= 0, 1, -1)          # 符号即类别

ada = AdaBoostScratch(n_estimators=50).fit(X_tr, y_tr)
print(f"从零 AdaBoost(50 树桩) test 准确率: {(ada.predict(X_te) == y_te).mean():.3f}")
print(f"单个树桩(对照)          test 准确率: {(DecisionTreeClassifier(max_depth=1).fit(X_tr,y_tr).predict(X_te)==y_te).mean():.3f}")
print("→ 50 个'弱'树桩串联远胜单个树桩 — boosting 把弱学习器叠成强模型")


<a id="3"></a>
## 3. 样本权重追难例（可视化）⭐ / Weights Chase the Hard Cases

AdaBoost 的灵魂是"**样本权重随轮次演化，越来越聚焦在难分的样本上**"。我们追踪一个**被反复分错的难样本**，看它的权重如何一轮轮被推高（模型越分不对它，它越被重视）；同时看每轮弱学习器的话语权 $\alpha$ 怎么随其错误率变化。
AdaBoost's soul is "**sample weights evolve over rounds, increasingly focusing on hard samples**". We track a **repeatedly-misclassified hard sample** and watch its weight pushed up round by round (the worse the model does on it, the more it's emphasized); we also watch each weak learner's say $\alpha$ versus its error.


In [ ]:
# 重训并记录: 每轮的弱学习器错误率、alpha, 以及某个难样本的权重 / track weight evolution
n = len(X_tr); w = np.full(n, 1/n)
errs, alphas, hard_weight = [], [], []
# 先找一个"难样本": 用一个小树预测错的样本 / pick a hard sample (misclassified by a small tree)
probe = DecisionTreeClassifier(max_depth=2).fit(X_tr, y_tr).predict(X_tr)
hard_idx = np.where(probe != y_tr)[0][0]

for _ in range(50):
    stump = DecisionTreeClassifier(max_depth=1).fit(X_tr, y_tr, sample_weight=w)
    pred = stump.predict(X_tr)
    err = min(max(w[pred != y_tr].sum(), 1e-10), 1-1e-10)
    alpha = 0.5*np.log((1-err)/err)
    w = w * np.exp(-alpha * y_tr * pred); w = w / w.sum()
    errs.append(err); alphas.append(alpha); hard_weight.append(w[hard_idx])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(errs, "o-", label="弱学习器加权错误率 err"); axes[0].plot(alphas, "s-", label="话语权 α")
axes[0].axhline(0.5, color="gray", ls=":", label="瞎猜 err=0.5(α=0)")
axes[0].set_xlabel("boosting 轮"); axes[0].legend(); axes[0].set_title("每轮: err 越低 → α(话语权) 越大")
axes[1].plot(np.array(hard_weight)/hard_weight[0], "o-", color="C3")
axes[1].set_xlabel("boosting 轮"); axes[1].set_ylabel("难样本权重(相对初始倍数)")
axes[1].set_title("一个难样本的权重: 被反复分错 → 权重被不断推高")
plt.tight_layout(); plt.show()
print("难样本权重随轮次被推高(模型越分不对它越重视它); err<0.5 → α>0(有正贡献)")


<a id="4"></a>
## 4. 指数损失 + 对异常值敏感 ⭐ / Exponential Loss & Outlier Sensitivity

AdaBoost 看似一堆启发式规则，其实有坚实的理论：**它在贪心地最小化指数损失** $L=\sum_i e^{-y_i F(\mathbf{x}_i)}$（其中 $F=\sum_m\alpha_m h_m$）。每加一个弱学习器、每次更新权重，都是这个损失的一步坐标下降——这是 AdaBoost 的统计学解释（Friedman 等）。
AdaBoost looks like a bag of heuristics but has solid theory: **it greedily minimizes the exponential loss** $L=\sum_i e^{-y_i F(\mathbf{x}_i)}$ (where $F=\sum_m\alpha_m h_m$). Each added learner and weight update is a coordinate-descent step on this loss — AdaBoost's statistical interpretation (Friedman et al.).

**一个重要后果（面试要点）**：指数损失对"被严重分错"的样本惩罚**呈指数增长**，所以 AdaBoost **对异常值/标签噪声很敏感**——一个错误标注的样本会被反复加权、主导训练。这正是 GBDT 常用更稳健的损失（如 log loss、Huber）的原因之一。
**An important consequence (interview point):** the exponential loss penalizes badly-misclassified samples **exponentially**, so AdaBoost is **sensitive to outliers/label noise** — a mislabeled sample gets repeatedly up-weighted and dominates training. This is one reason GBDT often uses more robust losses (log loss, Huber).


In [ ]:
# 对比各损失对"被严重分错"的惩罚 / how each loss penalizes margin y·F
m = np.linspace(-2, 2, 200)                              # margin = y·F (>0 分对, <0 分错)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(m, np.exp(-m), lw=2, label="指数损失 exp(-yF) — AdaBoost")
ax.plot(m, np.log2(1+np.exp(-m)), lw=2, label="对数损失 log(1+e⁻ʸᶠ) — GBDT/逻辑回归")
ax.plot(m, np.maximum(0, 1-m), lw=2, label="hinge — SVM")
ax.axvline(0, color="gray", lw=0.5); ax.set_xlabel("margin = y·F"); ax.set_ylabel("loss"); ax.legend()
ax.set_title("指数损失对'严重分错(margin 很负)'惩罚爆炸 → AdaBoost 对异常值敏感")
plt.tight_layout(); plt.show()
print("margin 很负(严重分错)时: 指数损失爆炸式增长, 对数损失只线性增长")
print("→ AdaBoost(指数损失)对异常值/标签噪声敏感; GBDT 可用更稳健的对数/Huber 损失")


<a id="5"></a>
## 5. 对照 sklearn + vs GBDT + 小结 ⭐ / sklearn, vs GBDT & Summary

对照 sklearn 的 `AdaBoostClassifier`，并和 GBDT（8.4）比较——两者都是 boosting，但**纠错方式不同**：
Compare with sklearn's `AdaBoostClassifier`, and contrast with GBDT (8.4) — both are boosting but **correct errors differently**:

| | AdaBoost | GBDT (8.4) |
|---|---|---|
| 怎么聚焦错误 how | **重加权样本**（难例权重↑）| **拟合残差/负梯度** |
| 损失 loss | 指数损失（固定）| 任意可导损失（灵活）|
| 对异常值 outliers | 敏感（指数惩罚）| 可稳健（换损失）|
| 关系 relation | GBDT 的特例/前身 | 更通用的框架 |


In [ ]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

print(f"{'模型':<26}{'CV 准确率':>10}")
for name, model in [("单个树桩 stump", DecisionTreeClassifier(max_depth=1)),
                    ("sklearn AdaBoost(50)", AdaBoostClassifier(n_estimators=50, random_state=0)),
                    ("GBDT(50)", GradientBoostingClassifier(n_estimators=50, random_state=0))]:
    print(f"{name:<28}{cross_val_score(model, X, y01, cv=5).mean():>10.3f}")
print("\nAdaBoost/GBDT 都把弱学习器叠成强模型; GBDT 拟合残差更通用, 是表格主流")
print("AdaBoost 历史意义大(第一个成功的 boosting), 实战已大多被 GBDT/XGBoost 取代")


```
boosting: 串行训练弱学习器, 每个补前面的错; 降偏差(vs bagging 并行降方差)
AdaBoost 算法: 初始权重均等 → 训弱学习器 → 算加权错误率 err 和话语权 α=½ln((1-err)/err)
              → 分错样本权重放大(×e^α)归一化 → 重复 → sign(Σα·h) 加权投票
样本权重"追难例": 反复分错的样本权重被不断推高, 逼后续学习器关注它
理论: AdaBoost = 贪心最小化指数损失 exp(-yF)
指数损失对严重分错惩罚爆炸 → AdaBoost 对异常值/标签噪声敏感(GBDT 可换稳健损失)
弱学习器(树桩)即可; AdaBoost 是 GBDT 的前身, 实战多被 GBDT/XGBoost 取代
```

### 💡 面试速查 / Interview cheat-sheet
1. **AdaBoost 串行 + 重加权难例**; 学习器话语权 α=½ln((1-err)/err)。
   AdaBoost is sequential + reweights hard cases; learner say α=½ln((1-err)/err).
2. **样本权重追难例**: 分错的权重放大, 逼下一个学习器关注它。
   Sample weights chase hard cases: misclassified ones amplified for the next learner.
3. **AdaBoost = 最小化指数损失**(理论解释)。
   AdaBoost = minimizing exponential loss (its theory).
4. **对异常值敏感**(指数惩罚爆炸); GBDT 可换稳健损失。
   Sensitive to outliers (exponential penalty explodes); GBDT can use robust losses.
5. **AdaBoost(重加权) vs GBDT(拟合残差)**; GBDT 更通用更主流。
   AdaBoost (reweighting) vs GBDT (residual-fitting); GBDT is more general and dominant.

### 下一节 / Next
**8.4 梯度提升推导**——把 boosting 推广: 不再重加权样本, 而是让每棵树**拟合损失的负梯度(伪残差)**, 这就是"函数空间的梯度下降"。
**8.4 Gradient Boosting Derivation** — generalize boosting: instead of reweighting samples, each tree **fits the negative gradient of the loss (pseudo-residual)** — "gradient descent in function space".
